# Notebook 7: Final comparison, statistics and figures

This notebook turns the per-method results of notebooks 3b to 6b into what goes in the thesis: **one final table**, **paired
statistical tests**, and **figures**. It answers Objective 4 of the proposal: *do the optimal transport and deeper alignment
methods give a real improvement over the lightweight baselines?*

No training and no GPU. It reads the `setting_B_metrics_*.csv` files that notebook 3b saved (and, for one figure, the saved
logits).

### What a statistical test can and cannot say here (read this first)
Every method is scored on the **same 10 grouped calibration/test splits**, so for each metric we have 10 *paired* numbers per
method. A paired test asks: *is the difference from the control consistently the same sign, larger than chance?*

* **What the 10 splits capture:** which images happened to be labelled for calibration, and (for the methods that are retrained
  per split) the randomness of that training.
* **What they do NOT capture:** how much the result would change if the **backbone** (the fine-tuned ResNet-50) were trained
  again with a different seed. That needs **several training seeds**, which we have not run.

So a "significant" result below means *"consistent across calibration splits for this one trained backbone"*, not *"would
reproduce with a new backbone"*. State this in the thesis. The last section shows what changes if you add more training seeds
(**a Wilcoxon test needs at least 6 independent units to be able to reach p < 0.05**).

### The tests we use
* **Paired Wilcoxon signed-rank test** (Wilcoxon, 1945; Demsar, 2006): compares two methods on paired scores without
  assuming they are normally distributed. With 10 pairs the smallest possible p-value is **0.002**, reached when all 10 splits agree.
* **Holm correction** (Holm, 1979): we make many comparisons, so some would look significant by luck. Holm adjusts the p-values
  within each metric across the methods. It is less conservative than Bonferroni.
* **Bootstrap 95% confidence interval** (Efron & Tibshirani, 1993) for the mean paired difference: resample the 10 splits with
  replacement many times and see how much the mean difference moves.
* We also report the **mean paired difference** (the effect size, which matters more than the p-value) and the **number of
  splits (out of 10) where the method wins**.

### Which comparison is the fair one?
Methods use different classifiers, so each is compared with its **matched control**, as in our earlier tables:

| Method | Matched control | Why |
|---|---|---|
| AdaBN | source_only | same network, only the batch-norm statistics change |
| Moment matching, JDOT | linear_head | a logistic regression on frozen features |
| Deep CORAL, JMMD, DANN, DeepJDOT | bottleneck_only | the same small bottleneck network without the alignment penalty |

We also compare every method with `source_only` (the headline baseline).

### Plan
1. Settings and loading
2. The final comparison table
3. Paired statistics against the matched controls
4. Paired statistics against `source_only`
5. Figures
6. If you have several training seeds
7. Summary

## Step 1: Settings and loading

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
# ---- Edit these -------------------------------------------------------------
OUT_DIR = Path("outputs")
FIG_DIR = OUT_DIR / "figures"
TRAIN_SEED = 0
SCORE_RULE = "MSP"                # confidence score for AUROC, AUPR and the risks: "MSP" or "best" (best score per method)
REFERENCE = "source_only"         # the headline baseline

N_BOOT = 10000                    # bootstrap resamples
ALPHA = 0.05                      # significance level (applied after the Holm correction)
RNG_SEED = 0

MANIFEST = OUT_DIR / "master_manifest.csv"          # only needed for the reliability-diagram figure
SPLIT_FOR_DIAGRAMS = 42
# -----------------------------------------------------------------------------

CONTROL_OF = {"adabn": "source_only", "moment_matching": "linear_head", "jdot": "linear_head",
              "deep_coral": "bottleneck_only", "jmmd": "bottleneck_only", "dann": "bottleneck_only", "deepjdot": "bottleneck_only"}
FAMILY = {"source_only": "control", "linear_head": "control", "bottleneck_only": "control",
          "adabn": "lightweight", "moment_matching": "lightweight", "dann": "lightweight",
          "deep_coral": "deep alignment", "jmmd": "deep alignment",
          "jdot": "optimal transport", "deepjdot": "optimal transport"}
COLORS = {"control": "#7f7f7f", "lightweight": "#1f77b4", "deep alignment": "#2ca02c", "optimal transport": "#d62728"}
ORDER = ["source_only", "linear_head", "bottleneck_only", "adabn", "moment_matching", "dann",
         "deep_coral", "jmmd", "jdot", "deepjdot"]

FIG_DIR.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(RNG_SEED)

In [ ]:
files = sorted(OUT_DIR.glob(f"setting_B_metrics_*_seed{TRAIN_SEED}.csv"))
results = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

present = list(results["method"].unique())
method_order = [m for m in ORDER if m in present] + sorted(m for m in present if m not in ORDER)
print("Methods found:", method_order)

# Every method must have been scored on all splits, on the same test images
assert results.groupby("method")["split_seed"].nunique().nunique() == 1, "Methods have different numbers of splits"
assert results.groupby("split_seed")["n_test"].nunique().eq(1).all(), "Methods were scored on different test sets"
n_splits = results["split_seed"].nunique()
print(n_splits, "splits per method")

We reduce each method's wide table to the metrics we compare. For AUROC, AUPR and the risks we use the confidence score chosen by
`SCORE_RULE`: **MSP for everyone** (the like-for-like choice), or the best score per method (the paper's convention).
`ConfGap_raw` is the raw reliability gap (mean confidence minus accuracy, before temperature scaling).

In [ ]:
SCORES = ["MSP", "Entropy", "Margin", "Energy"]
METRICS = ["Accuracy", "ECE_cal", "Brier_cal", "AUROC_unknown", "AUPR_unknown", "Risk@80", "Risk@50", "ConfGap_raw"]
HIGHER_IS_BETTER = {"Accuracy": True, "AUROC_unknown": True, "AUPR_unknown": True,
                    "ECE_cal": False, "Brier_cal": False, "Risk@80": False, "Risk@50": False, "ConfGap_raw": False}


def per_split_table(results, rule):
    parts = []
    for method, table in results.groupby("method"):
        score = max(SCORES, key=lambda s: table[f"AUROC_{s}"].mean()) if rule == "best" else rule
        parts.append(pd.DataFrame({
            "method": method, "split_seed": table["split_seed"].values, "score": score,
            "Accuracy": table["Accuracy"].values, "ECE_cal": table["ECE_cal"].values, "Brier_cal": table["Brier_cal"].values,
            "AUROC_unknown": table[f"AUROC_{score}"].values, "AUPR_unknown": table[f"AUPR_{score}"].values,
            "Risk@80": table[f"Risk@80_{score}"].values, "Risk@50": table[f"Risk@50_{score}"].values,
            "ConfGap_raw": table["ConfGap_raw"].values, "T": table["T"].values,
        }))
    return pd.concat(parts, ignore_index=True)


frame = per_split_table(results, SCORE_RULE)
frame.groupby("method").size().to_dict()

## Step 2: The final comparison table

Mean and standard deviation over the 10 splits. The standard deviation here is the spread **between splits**, not between
trained models.

In [ ]:
mean = frame.groupby("method")[METRICS].mean().loc[method_order]
std = frame.groupby("method")[METRICS].std().loc[method_order]

final_table = mean.round(4).astype(str) + " ± " + std.round(4).astype(str)
final_table.insert(0, "family", [FAMILY.get(m, "") for m in method_order])
final_table.to_csv(OUT_DIR / "final_table_settingB.csv")
final_table

## Step 3: Paired statistics against the matched controls

Four small functions do all the work:
* `wilcoxon_p`: the paired Wilcoxon p-value for the 10 differences;
* `bootstrap_ci`: the 95% interval of their mean;
* `holm`: the Holm adjustment of a list of p-values;
* `paired_table`: runs them for every method and every metric.

A difference is computed as **method minus control**, split by split, so *positive* means the method scored higher. Whether that
is good depends on the metric (higher accuracy is good, higher risk is bad), so the table also marks the direction.

In [ ]:
def wilcoxon_p(diff):
    if np.allclose(diff, 0):
        return 1.0                                           # identical scores: nothing to test
    return float(stats.wilcoxon(diff, zero_method="wilcox", alternative="two-sided").pvalue)


def bootstrap_ci(values, n_boot=N_BOOT):
    values = np.asarray(values, dtype=float)
    picks = rng.integers(0, len(values), size=(n_boot, len(values)))     # resample the splits with replacement
    means = values[picks].mean(axis=1)
    return tuple(np.percentile(means, [2.5, 97.5]))


def holm(p_values):
    # Holm step-down adjustment (the smallest p is multiplied by m, the next by m-1, ...)
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    m = len(p)
    adjusted = np.minimum(1.0, np.maximum.accumulate((m - np.arange(m)) * p[order]))
    out = np.empty(m)
    out[order] = adjusted
    return out

**Check the helpers on cases with known answers** before trusting them: a difference that is the same on all 10 splits should
give the smallest possible p-value (0.00195), pure noise should give a large one, and Holm should multiply the smallest of
three p-values by 3.

In [ ]:
print("all 10 splits agree:     p =", round(wilcoxon_p(np.arange(1.0, 11.0)), 5), "(expect 0.00195)")
print("pure noise:              p =", round(wilcoxon_p(np.array([1, -2, 3, -4, 5, -6, 7, -8, 9, -10.0])), 3), "(expect large)")
print("Holm of [0.01, 0.02, 0.5] =", np.round(holm([0.01, 0.02, 0.5]), 3), "(expect [0.03, 0.04, 0.5])")
low, high = bootstrap_ci(np.full(10, 2.0))
print("bootstrap CI of a constant 2.0:", (round(low, 3), round(high, 3)), "(expect (2.0, 2.0))")

In [ ]:
def paired_table(frame, pairs, metrics=METRICS):
    # pairs: {method: control}. One row per (method, metric).
    rows = []
    for method, control in pairs.items():
        a = frame[frame["method"] == method].set_index("split_seed")
        b = frame[frame["method"] == control].set_index("split_seed")
        for metric in metrics:
            diff = (a[metric] - b[metric]).dropna()                   # method minus control, split by split
            low, high = bootstrap_ci(diff.values)
            better = (diff > 0) if HIGHER_IS_BETTER[metric] else (diff < 0)
            rows.append({"method": method, "control": control, "metric": metric,
                         "mean difference": diff.mean(), "ci_low": low, "ci_high": high,
                         "splits better": int(better.sum()), "p": wilcoxon_p(diff.values)})
    table = pd.DataFrame(rows)

    # Holm correction within each metric, across the methods being compared
    table["p_holm"] = table.groupby("metric")["p"].transform(lambda p: pd.Series(holm(p.values), index=p.index))

    def verdict(row):
        significant = row["p_holm"] < ALPHA and (row["ci_low"] > 0 or row["ci_high"] < 0)
        if not significant:
            return "="                                                # no reliable difference
        improved = (row["mean difference"] > 0) == HIGHER_IS_BETTER[row["metric"]]
        return "better" if improved else "worse"

    table["verdict"] = table.apply(verdict, axis=1)
    return table

### Against the matched controls (the primary comparison)

In [ ]:
primary_pairs = {m: c for m, c in CONTROL_OF.items() if m in method_order and c in method_order}
primary = paired_table(frame, primary_pairs)

def readable(table):
    # One cell per method and metric: the mean difference and a verdict marker (+ = significantly better, - = significantly worse)
    marker = {"better": " (+)", "worse": " (-)", "=": ""}
    cells = table.assign(cell=table["mean difference"].map(lambda v: f"{v:+.3f}") + table["verdict"].map(marker))
    return cells.pivot(index="method", columns="metric", values="cell")[METRICS].loc[[m for m in method_order if m in table["method"].unique()]]

readable(primary)

**How to read it.** Each cell is the **mean paired difference from the matched control** (method minus control).
`(+)` means significantly **better** and `(-)` significantly **worse**, after the Holm correction **and** with a bootstrap interval that
excludes zero. No marker means we cannot tell it apart from noise. Remember the direction: a negative number is good for
ECE, Brier, the risks and the raw gap, and a positive number is good for accuracy, AUROC and AUPR.

The full table, with confidence intervals and exact p-values, is below.

In [ ]:
detail = primary.assign(**{"mean difference": primary["mean difference"].round(4), "ci_low": primary["ci_low"].round(4),
                           "ci_high": primary["ci_high"].round(4), "p": primary["p"].round(4), "p_holm": primary["p_holm"].round(4)})
detail.to_csv(OUT_DIR / "stats_vs_matched_controls.csv", index=False)
detail[detail["metric"].isin(["Accuracy", "Risk@80", "AUROC_unknown", "ConfGap_raw"])].sort_values(["metric", "method"])

## Step 4: Paired statistics against `source_only`

The same analysis, with the headline baseline as the reference for **every** method (the two other controls included, so you can
see what the frozen-feature pipeline itself does).

In [ ]:
reference_pairs = {m: REFERENCE for m in method_order if m != REFERENCE}
against_reference = paired_table(frame, reference_pairs)
against_reference.round(4).to_csv(OUT_DIR / "stats_vs_source_only.csv", index=False)
readable(against_reference)

**Reading the two tables together.** A method that is significantly better than `source_only` but **not** better than its matched
control is gaining from the change of *classifier*, not from the adaptation. The adaptation itself is what the first table
(against the matched control) measures.

## Step 5: Figures

All figures are saved as PNG files in `outputs/figures/`.

### Figure 1: accuracy with 95% intervals
Each dot is a method's mean accuracy over the 10 splits, with a bootstrap 95% interval. Grey dots are the controls.

In [ ]:
def style(method):
    return COLORS[FAMILY.get(method, "control")]


order = frame.groupby("method")["Accuracy"].mean().sort_values().index.tolist()
fig, ax = plt.subplots(figsize=(7.5, 5))
for y, method in enumerate(order):
    values = frame.loc[frame["method"] == method, "Accuracy"].values
    low, high = bootstrap_ci(values)
    ax.errorbar(values.mean(), y, xerr=[[values.mean() - low], [high - values.mean()]], fmt="o", color=style(method), capsize=3)
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order)
ax.set_xlabel("accuracy on the field test images (mean over 10 splits, 95% bootstrap interval)")
ax.axvline(frame.loc[frame["method"] == REFERENCE, "Accuracy"].mean(), color="k", linestyle=":", linewidth=1)
handles = [plt.Line2D([], [], marker="o", linestyle="", color=c, label=f) for f, c in COLORS.items()]
ax.legend(handles=handles, loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig1_accuracy.png", dpi=200)
plt.show()

### Figure 2: the trade-off between accuracy and unknown-disease detection
Each point is a method. The horizontal axis is its change in **unknown AUROC** and the vertical axis its change in
**accuracy**, both relative to its matched control. Points in the **upper left** gain accuracy but lose the ability to flag unknown
diseases. This is the central pattern of our results.

In [ ]:
acc = primary[primary["metric"] == "Accuracy"].set_index("method")["mean difference"]
auc = primary[primary["metric"] == "AUROC_unknown"].set_index("method")["mean difference"]

fig, ax = plt.subplots(figsize=(6.5, 5.5))
for method in acc.index:
    ax.scatter(auc[method], acc[method], s=70, color=style(method))
    ax.annotate(method, (auc[method], acc[method]), textcoords="offset points", xytext=(6, 4), fontsize=8)
ax.axhline(0, color="k", linewidth=0.8)
ax.axvline(0, color="k", linewidth=0.8)
ax.set_xlabel("change in unknown AUROC vs the matched control (right = better)")
ax.set_ylabel("change in accuracy vs the matched control (up = better)")
ax.set_title("Accuracy gained vs unknown detection lost", fontsize=10)
ax.legend(handles=[plt.Line2D([], [], marker="o", linestyle="", color=c, label=f) for f, c in COLORS.items() if f != "control"], fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig2_tradeoff.png", dpi=200)
plt.show()

### Figure 3: the raw reliability gap
Mean confidence minus accuracy **before** temperature scaling (0 would be honest). Bars show the spread between splits.
Remember that part of a lower gap can come from smaller logits, which a single temperature removes.

In [ ]:
gap_mean = frame.groupby("method")["ConfGap_raw"].mean().loc[method_order]
gap_std = frame.groupby("method")["ConfGap_raw"].std().loc[method_order]

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.barh(range(len(method_order)), gap_mean.values, xerr=gap_std.values, color=[style(m) for m in method_order], capsize=3)
ax.set_yticks(range(len(method_order)))
ax.set_yticklabels(method_order)
ax.invert_yaxis()
ax.set_xlabel("raw reliability gap: mean confidence - accuracy (lower is better)")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig3_raw_gap.png", dpi=200)
plt.show()

### Figure 4: significance map
A compact picture of the first statistics table. Green = significantly better than the matched control, red = significantly
worse, grey = no reliable difference. The number in each cell is the mean paired difference.

In [ ]:
methods_shown = [m for m in method_order if m in primary["method"].unique()]
code_of = {"better": 1, "=": 0, "worse": -1}
grid = np.array([[code_of[primary[(primary["method"] == m) & (primary["metric"] == k)]["verdict"].iloc[0]] for k in METRICS] for m in methods_shown])
values = np.array([[primary[(primary["method"] == m) & (primary["metric"] == k)]["mean difference"].iloc[0] for k in METRICS] for m in methods_shown])

from matplotlib.colors import ListedColormap
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.imshow(grid, cmap=ListedColormap(["#e6a1a1", "#e0e0e0", "#a1d9a1"]), vmin=-1, vmax=1, aspect="auto")
for i in range(len(methods_shown)):
    for j in range(len(METRICS)):
        ax.text(j, i, f"{values[i, j]:+.3f}", ha="center", va="center", fontsize=7)
ax.set_xticks(range(len(METRICS)))
ax.set_xticklabels([k + (" ↑" if HIGHER_IS_BETTER[k] else " ↓") for k in METRICS], rotation=30, ha="right", fontsize=8)
ax.set_yticks(range(len(methods_shown)))
ax.set_yticklabels([f"{m}  (vs {primary_pairs[m]})" for m in methods_shown], fontsize=8)
ax.set_title("Paired comparison with each method's matched control (green better, red worse, grey no reliable difference)", fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig4_significance_map.png", dpi=200)
plt.show()

### Figure 5: reliability diagrams (raw, before temperature scaling)
For one split, the bars show the **actual accuracy** in each confidence bin and the dotted line is perfect calibration. This
needs the saved logits, and it skips any method whose file is missing. Faint bars hold fewer than 20 images.

In [ ]:
manifest = pd.read_csv(MANIFEST, keep_default_na=False)
known_rows = manifest[(manifest["dataset"] == "pd") & (manifest["role"] == "pd_known")].set_index("path")

def load_logits(method, seed):
    f = np.load(OUT_DIR / f"outputs_{method}_seed{TRAIN_SEED}.npz", allow_pickle=True)
    key = f"pd_known_logits_B_seed{seed}"
    return (f[key] if key in f.files else f["pd_known_logits"]), f["pd_known_labels"], f["pd_known_paths"]


def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def reliability_bars(ax, confidence, correct, title, n_bins=15, min_count=20):
    edges = np.linspace(0, 1, n_bins + 1)
    for low, high in zip(edges[:-1], edges[1:]):
        in_bin = (confidence > low) & (confidence <= high)
        if in_bin.any():
            ax.bar(confidence[in_bin].mean(), correct[in_bin].mean(), width=0.9 / n_bins, color="C0",
                   alpha=0.85 if in_bin.sum() >= min_count else 0.25)
    ax.plot([0, 1], [0, 1], "k:")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title(title, fontsize=9)


diagram_methods = [m for m in ["source_only", "bottleneck_only", "adabn", "deep_coral", "jdot", "deepjdot"]
                   if (OUT_DIR / f"outputs_{m}_seed{TRAIN_SEED}.npz").exists()]
fig, axes = plt.subplots(2, 3, figsize=(12, 7), sharex=True, sharey=True)
for ax, method in zip(axes.ravel(), diagram_methods):
    logits, labels, paths = load_logits(method, SPLIT_FOR_DIAGRAMS)
    is_test = (known_rows.loc[paths, f"B_seed{SPLIT_FOR_DIAGRAMS}"] == "test").to_numpy()
    probs = softmax(logits[is_test])
    confidence, correct = probs.max(axis=1), (probs.argmax(axis=1) == labels[is_test]).astype(float)
    reliability_bars(ax, confidence, correct, f"{method}  (mean conf {confidence.mean():.2f}, acc {correct.mean():.2f})")
for ax in axes.ravel()[len(diagram_methods):]:
    ax.axis("off")
fig.supxlabel("confidence")
fig.supylabel("accuracy in the bin")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig5_reliability_diagrams.png", dpi=200)
plt.show()

## Step 6: If you have several training seeds

Everything above is for one trained backbone (`TRAIN_SEED = 0`). To make a claim that would survive re-training you need results
from several training seeds (re-run notebooks 02, 02b and the method notebooks with `TRAIN_SEED = 1, 2, ...`, and score them
in 3b). Then each **training seed** is one independent unit.

This cell looks for other seeds' result files. If it finds some, it averages each method's metric over the splits within each
seed and shows the per-seed values. **How many seeds do you need for a test?** The smallest possible Wilcoxon p-value is 0.25 with 3
units, 0.0625 with 5 and **0.031 with 6**. So with fewer than 6 seeds, a test *cannot* be significant, and the honest thing is to
show the per-seed numbers and whether the sign agrees in every seed.

In [ ]:
seed_files = sorted(OUT_DIR.glob("setting_B_metrics_*_seed*.csv"))
all_seeds = pd.concat([pd.read_csv(f) for f in seed_files], ignore_index=True)
training_seeds = sorted(int(s) for s in all_seeds["train_seed"].unique())
print("Training seeds found:", training_seeds)

if len(training_seeds) < 2:
    print("Only one training seed so far, so the tests above are about calibration-split variation only.")
else:
    by_seed = (per_split_table(all_seeds.assign(method=all_seeds["method"] + "||" + all_seeds["train_seed"].astype(str)), SCORE_RULE))
    by_seed[["method", "seed"]] = by_seed["method"].str.split("||", regex=False, expand=True)
    seed_means = by_seed.groupby(["method", "seed"])["Accuracy"].mean().unstack("seed").loc[[m for m in method_order if m in by_seed["method"].unique()]]
    print("Mean accuracy per training seed (each seed averaged over its 10 splits):")
    display(seed_means.round(4))
    needed = 6
    print(f"\nWith {len(training_seeds)} seeds a Wilcoxon test {'can' if len(training_seeds) >= needed else 'cannot'} reach p < 0.05 (needs at least {needed}).")

## Step 7: Summary

In [ ]:
n_tests = len(primary)
significant = primary[primary["verdict"] != "="]

print(f"Comparisons against the matched controls: {n_tests}")
print(f"  significantly better: {int((primary['verdict'] == 'better').sum())}")
print(f"  significantly worse:  {int((primary['verdict'] == 'worse').sum())}")
print(f"  no reliable difference: {int((primary['verdict'] == '=').sum())}")
print()

for metric in ["Accuracy", "Risk@80", "AUROC_unknown", "ConfGap_raw"]:
    sub = primary[primary["metric"] == metric]
    better = sub.loc[sub["verdict"] == "better", "method"].tolist()
    worse = sub.loc[sub["verdict"] == "worse", "method"].tolist()
    print(f"{metric:14s} better: {better if better else 'none'} | worse: {worse if worse else 'none'}")

### What you can write in the thesis
* *"Each method was compared with its matched control on the same 10 grouped calibration/test splits using a paired Wilcoxon
  signed-rank test, Holm-corrected within each metric, together with a bootstrap 95% interval for the mean paired difference."*
* *"The tests quantify consistency across calibration splits for a single trained backbone. They do not capture backbone-training
  variance, which we leave to future work."* (or, if you add seeds: *"... and across N training seeds."*)
* Report **effect sizes** (the mean differences and their intervals), not just stars.

### Next
**Notebook 7b** adds **Grad-CAM**: where is the network looking on lab images versus field images, and does adaptation change it?